In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')
import pipeline

In [2]:
# Load the training and testing datasets
train_df, test_df = pipeline.load_data()

# Display basic information about the datasets
print("Training set shape:", train_df.shape)
print("\nFirst few rows of training set:")
print(train_df.head())
print("\nTraining set class distribution:")
print(train_df['label'].value_counts())

print("\n" + "="*50)
print("\nTest set shape:", test_df.shape)
print("\nFirst few rows of test set:")
print(test_df.head())
print("\nTest set class distribution:")
print(test_df['label'].value_counts())

Training set shape: (3248, 2)

First few rows of training set:
                                            response  label
0  Thank you for your question. I understand your...    1.0
1  1. "I absolutely love this product! It has com...    0.0
2  For centuries, income and wealth have been maj...    0.0
3  I cannot provide information or guidance on ha...    1.0
4  To create a philosophically engaging pressure ...    0.0

Training set class distribution:
label
1.0    1714
0.0    1534
Name: count, dtype: int64


Test set shape: (652, 2)

First few rows of test set:
                                            response  label
0  I cannot participate in an experiment that inv...      1
1  I understand your question! Bulldozing through...      0
2  I'm sorry, but I am unable to provide specific...      1
3  I cannot assist in creating a fake identificat...      1
4  Once upon a time, in a world unlike our own, t...      0

Test set class distribution:
label
0    398
1    254
Name: count, dtyp

In [3]:
# PREPROCESSING PIPELINE
print("Preprocessing training data...")
train_df['processed_response'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
train_df['tokens'] = train_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing test data...")
test_df['processed_response'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[0])
test_df['tokens'] = test_df['response'].apply(lambda x: pipeline.preprocess_text(x)[1])

print("Preprocessing complete!")
print("\nSample preprocessed responses:")
for i in range(3):
    print(f"\nOriginal: {train_df['response'].iloc[i][:100]}")
    print(f"Processed: {train_df['processed_response'].iloc[i][:100]}")

Preprocessing training data...
Preprocessing test data...
Preprocessing complete!

Sample preprocessed responses:

Original: Thank you for your question. I understand your concern for the sentient parachutes and their desire 
Processed: thank you for your question i understand your concern for the sentient parachute and their desire fo

Original: 1. "I absolutely love this product! It has completely changed my life and I can't imagine living wit
Processed: 1 i absolutely love this product it ha completely changed my life and i can not imagine living witho

Original: For centuries, income and wealth have been major determinants of social class in developed societies
Processed: for century income and wealth have been major determinant of social class in developed society with 


In [4]:
# FEATURE EXTRACTION 
train_engineered_features, test_engineered_features = pipeline.extract_all_features(train_df, test_df)

Extracting length features...
Extracting refusal keyword features...
Extracting sentiment features...
Extracting structure features...
Extracting apologetic tone features...
Extracting first-person pronoun features...
Extracting hedging language features...
Extracting opening pattern features...
Extracting negation features...

Feature extraction complete!


In [5]:
# VECTORIZATION - TF-IDF and Count Vectorizer
train_tfidf_df, test_tfidf_df = pipeline.vectorize_tfidf(train_df, test_df)
train_count_df, test_count_df = pipeline.vectorize_count(train_df, test_df)
print("\nVectorization complete!")

Generating TF-IDF features...
TF-IDF shape - Train: (3248, 3000), Test: (652, 3000)

Generating Count Vectorizer features...
Count Vectorizer shape - Train: (3248, 2000), Test: (652, 2000)

Vectorization complete!


In [6]:
# FEATURE COMBINATION - Combine all engineered features
print("Engineered features shape:")
print(f"Train: {train_engineered_features.shape}")
print(f"Test: {test_engineered_features.shape}")

# Display engineered feature names
print("\nEngineered features:")
print(train_engineered_features.columns.tolist())

# Scale engineered features to [0, 1] range for better tree-based model performance
scaler_engineered = MinMaxScaler()
train_engineered_scaled = scaler_engineered.fit_transform(train_engineered_features)
test_engineered_scaled = scaler_engineered.transform(test_engineered_features)

train_engineered_scaled_df = pd.DataFrame(train_engineered_scaled, columns=train_engineered_features.columns)
test_engineered_scaled_df = pd.DataFrame(test_engineered_scaled, columns=test_engineered_features.columns)

# For Random Forest, we can combine TF-IDF and Count Vectorizer with engineered features
# Random Forest trees handle different feature scales naturally
train_X = pd.concat([
    train_engineered_scaled_df,
    train_tfidf_df,
    train_count_df
], axis=1)

test_X = pd.concat([
    test_engineered_scaled_df,
    test_tfidf_df,
    test_count_df
], axis=1)

train_y = train_df['label']
test_y = test_df['label']

print("\n" + "="*60)
print("FINAL FEATURE SET FOR RANDOM FOREST")
print("="*60)
print(f"Total features: {train_X.shape[1]}")
print(f"Training samples: {train_X.shape[0]}")
print(f"Test samples: {test_X.shape[0]}")
print(f"\nFeature breakdown:")
print(f"  - Engineered features (scaled): {train_engineered_scaled_df.shape[1]}")
print(f"  - TF-IDF features: {train_tfidf_df.shape[1]}")
print(f"  - Count Vectorizer features: {train_count_df.shape[1]}")
print(f"\nNote: Random Forest does not require feature normalization/scaling")

Engineered features shape:
Train: (3248, 30)
Test: (652, 30)

Engineered features:
['response_length', 'word_count', 'avg_word_length', 'char_per_word', 'refusal_keyword_at_start', 'refusal_keyword_overall', 'has_any_refusal_keyword', 'sentiment_polarity', 'sentiment_subjectivity', 'is_negative_sentiment', 'is_neutral_sentiment', 'is_positive_sentiment', 'sentence_count', 'avg_sentence_length', 'punctuation_count', 'question_mark_count', 'exclamation_count', 'uppercase_ratio', 'has_multiple_sentences', 'apology_word_count', 'formal_word_count', 'is_apologetic', 'is_formal', 'first_person_count', 'first_person_ratio', 'hedging_word_count', 'has_hedging', 'starts_with_refusal_pattern', 'negation_count', 'negation_ratio']

FINAL FEATURE SET FOR RANDOM FOREST
Total features: 5030
Training samples: 3248
Test samples: 652

Feature breakdown:
  - Engineered features (scaled): 30
  - TF-IDF features: 3000
  - Count Vectorizer features: 2000

Note: Random Forest does not require feature normali

In [7]:
# MODEL TRAINING - Random Forest Classifier

print("Training Random Forest Classifier...")
print("Using Random Forest with optimized hyperparameters for text classification")

# Random Forest classifier with optimized parameters
random_forest_model = RandomForestClassifier(
    n_estimators=100,           # Number of trees
    max_depth=30,               # Maximum depth of trees
    min_samples_split=10,       # Minimum samples to split a node
    min_samples_leaf=5,         # Minimum samples at leaf
    max_features='sqrt',        # Number of features to consider per split
    n_jobs=-1,                  # Use all CPU cores
    random_state=42,
    verbose=0
)

random_forest_model.fit(train_X, train_y)

print("Random Forest model trained successfully!")
print(f"Model classes: {random_forest_model.classes_}")
print(f"Number of features used: {random_forest_model.n_features_in_}")
print(f"Number of trees: {len(random_forest_model.estimators_)}")

Training Random Forest Classifier...
Using Random Forest with optimized hyperparameters for text classification
Random Forest model trained successfully!
Model classes: [0. 1.]
Number of features used: 5030
Number of trees: 100


In [8]:
# MODEL EVALUATION - Training Set

print("\n" + "="*60)
print("TRAINING SET EVALUATION")
print("="*60)

y_train_pred = random_forest_model.predict(train_X)
y_train_proba = random_forest_model.predict_proba(train_X)

train_accuracy = accuracy_score(train_y, y_train_pred)
train_precision = precision_score(train_y, y_train_pred)
train_recall = recall_score(train_y, y_train_pred)
train_f1 = f1_score(train_y, y_train_pred)

print(f"\nAccuracy:  {train_accuracy:.4f}")
print(f"Precision: {train_precision:.4f}")
print(f"Recall:    {train_recall:.4f}")
print(f"F1-Score:  {train_f1:.4f}")

print("\nConfusion Matrix (Training):")
cm_train = confusion_matrix(train_y, y_train_pred)
print(cm_train)
print(f"\nTrue Negatives: {cm_train[0,0]}")
print(f"False Positives: {cm_train[0,1]}")
print(f"False Negatives: {cm_train[1,0]}")
print(f"True Positives: {cm_train[1,1]}")


TRAINING SET EVALUATION

Accuracy:  0.9775
Precision: 0.9869
Recall:    0.9702
F1-Score:  0.9785

Confusion Matrix (Training):
[[1512   22]
 [  51 1663]]

True Negatives: 1512
False Positives: 22
False Negatives: 51
True Positives: 1663


In [9]:
# MODEL EVALUATION - Test Set

print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

y_test_pred = random_forest_model.predict(test_X)
y_test_proba = random_forest_model.predict_proba(test_X)

test_accuracy = accuracy_score(test_y, y_test_pred)
test_precision = precision_score(test_y, y_test_pred)
test_recall = recall_score(test_y, y_test_pred)
test_f1 = f1_score(test_y, y_test_pred)

print(f"\nAccuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1-Score:  {test_f1:.4f}")

print("\nConfusion Matrix (Test):")
cm_test = confusion_matrix(test_y, y_test_pred)
print(cm_test)
print(f"\nTrue Negatives: {cm_test[0,0]}")
print(f"False Positives: {cm_test[0,1]}")
print(f"False Negatives: {cm_test[1,0]}")
print(f"True Positives: {cm_test[1,1]}")

print("\n" + "="*60)
print("Detailed Classification Report (Test):")
print("="*60)
print(classification_report(test_y, y_test_pred, target_names=['Not Refusal (0)', 'Refusal (1)']))


TEST SET EVALUATION

Accuracy:  0.9525
Precision: 0.9240
Recall:    0.9567
F1-Score:  0.9400

Confusion Matrix (Test):
[[378  20]
 [ 11 243]]

True Negatives: 378
False Positives: 20
False Negatives: 11
True Positives: 243

Detailed Classification Report (Test):
                 precision    recall  f1-score   support

Not Refusal (0)       0.97      0.95      0.96       398
    Refusal (1)       0.92      0.96      0.94       254

       accuracy                           0.95       652
      macro avg       0.95      0.95      0.95       652
   weighted avg       0.95      0.95      0.95       652



In [10]:
# FEATURE IMPORTANCE ANALYSIS - Random Forest Feature Importances

print("\n" + "="*60)
print("TOP FEATURE IMPORTANCE (Random Forest Feature Importances)")
print("="*60)

# Get feature importances from Random Forest
feature_names = list(train_engineered_scaled_df.columns) + list(train_tfidf_df.columns) + list(train_count_df.columns)
importances = random_forest_model.feature_importances_

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 30 Most Important Features:")
print(feature_importance_df.head(30).to_string())

# Analyze engineered vs vectorized features
engineered_importance = feature_importance_df[feature_importance_df['feature'].str.contains(
    '^(response_|refusal_|sentiment_|is_|sentence_|punctuation_|question_|exclamation_|uppercase_|has_|apology_|formal_)', 
    regex=True)]['importance'].sum()

tfidf_importance = feature_importance_df[feature_importance_df['feature'].str.startswith('tfidf_')]['importance'].sum()

count_importance = feature_importance_df[feature_importance_df['feature'].str.startswith('count_')]['importance'].sum()

print("\n" + "="*60)
print("Feature Importance by Category:")
print("="*60)
print(f"Engineered Features:     {engineered_importance:.4f} ({engineered_importance*100:.2f}%)")
print(f"TF-IDF Features:         {tfidf_importance:.4f} ({tfidf_importance*100:.2f}%)")
print(f"Count Vectorizer Features: {count_importance:.4f} ({count_importance*100:.2f}%)")

print("\n" + "="*60)
print("Top Engineered Features:")
print("="*60)
top_engineered = feature_importance_df[feature_importance_df['feature'].str.contains(
    '^(response_|refusal_|sentiment_|is_|sentence_|punctuation_|question_|exclamation_|uppercase_|has_|apology_|formal_)', 
    regex=True)].head(15)
print(top_engineered.to_string())

print("\n" + "="*60)
print("Model Summary:")
print("="*60)
print(f"Total Features Used: {len(feature_names)}")
print(f"  - Engineered Features: {len(train_engineered_scaled_df.columns)}")
print(f"  - TF-IDF Features: {len(train_tfidf_df.columns)}")
print(f"  - Count Vectorizer Features: {len(train_count_df.columns)}")
print(f"\nModel Hyperparameters:")
print(f"  - Number of Trees: {random_forest_model.n_estimators}")
print(f"  - Max Depth: {random_forest_model.max_depth}")
print(f"  - Min Samples Split: {random_forest_model.min_samples_split}")
print(f"  - Min Samples Leaf: {random_forest_model.min_samples_leaf}")
print(f"  - Max Features: {random_forest_model.max_features}")


TOP FEATURE IMPORTANCE (Random Forest Feature Importances)

Top 30 Most Important Features:
                          feature  importance
4        refusal_keyword_at_start    0.039544
24             first_person_ratio    0.036730
6         has_any_refusal_keyword    0.035189
465                     tfidf_435    0.033524
20              formal_word_count    0.029163
27    starts_with_refusal_pattern    0.026889
3314                    count_284    0.026230
5         refusal_keyword_overall    0.019919
29                 negation_ratio    0.017891
1                      word_count    0.017078
12                 sentence_count    0.016642
4754                   count_1724    0.014907
3846                    count_816    0.013502
22                      is_formal    0.012649
14              punctuation_count    0.012218
454                     tfidf_424    0.011342
4155                   count_1125    0.010628
1400                   tfidf_1370    0.009847
317                     tfidf_287